# Normattiva OpenData API — Test Notebook

Questo notebook testa tutte le API documentate nelle *Specifiche API Open Data* di Normattiva (IPZS, rev. 08 — 30/10/2025).

**Ambienti disponibili:**
- **PRE (test):** `https://pre.api.normattiva.it/t/normattiva.api`
- **PROD:** `https://api.normattiva.it/t/normattiva.api`

---
**Struttura del notebook:**
1. Setup & configurazione
2. Tipologiche (dati di riferimento)
3. Ricerca sincrona (semplice, avanzata, con FacetMap)
4. Ricerca asincrona (export ZIP)
5. Dettaglio atto
6. Download collezione preconfezionata
7. Atti aggiornati tra due date

---
## 0. Setup & configurazione

In [1]:
import requests
import json
import time
from pprint import pprint
from datetime import datetime, timedelta

import ssl
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context


class LegacyTLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = create_urllib3_context()
        ctx.options |= ssl.OP_LEGACY_SERVER_CONNECT
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)


SESSION = requests.Session()
SESSION.mount("https://", LegacyTLSAdapter())

# ── Ambiente ──────────────────────────────────────────────────────────────────
ENV = "PROD"  # cambiare in "PRE" per l'ambiente di test (richiede certificato client mTLS)

BASE_URLS = {
    "PRE":  "https://pre.api.normattiva.it/t/normattiva.api",
    "PROD": "https://api.normattiva.it/t/normattiva.api",
}
BASE_URL = BASE_URLS[ENV]
print(f"Ambiente selezionato: {ENV}")
print(f"Base URL: {BASE_URL}")

Ambiente selezionato: PROD
Base URL: https://api.normattiva.it/t/normattiva.api


In [2]:
# ── Headers comuni ────────────────────────────────────────────────────────────
HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive",
    "Origin": "https://pre.dati.normattiva.it" if ENV == "PRE" else "https://dati.normattiva.it",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
}
HEADERS_JSON = {**HEADERS, "Content-Type": "application/json"}


# ── Funzioni di utilità ───────────────────────────────────────────────────────
def get(path, params=None, expect_binary=False):
    """GET con stampa status e (opzionale) body JSON."""
    url = BASE_URL + path
    r = SESSION.get(url, headers=HEADERS, params=params)
    print(f"GET {url}  →  {r.status_code}")
    if not expect_binary:
        try:
            return r.status_code, r.json()
        except Exception:
            return r.status_code, r.text
    return r.status_code, r.content


def post(path, body):
    """POST JSON con stampa status e body JSON."""
    url = BASE_URL + path
    r = SESSION.post(url, headers=HEADERS_JSON, json=body)
    print(f"POST {url}  →  {r.status_code}")
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, r.text


def put(path, body):
    """PUT JSON con stampa status e body JSON."""
    url = BASE_URL + path
    r = SESSION.put(url, headers=HEADERS_JSON, json=body)
    print(f"PUT {url}  →  {r.status_code}")
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, r.text


def pp(data):
    """Pretty-print JSON."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

print("Setup completato.")

Setup completato.


---
## 1. Tipologiche (dati di riferimento)

Queste API restituiscono liste di valori ammissibili da usare come input nelle API successive.

### 1.1 Estensioni — formati di esportazione

`GET /bff-opendata/v1/api/v1/tipologiche/estensioni`

Recupera tutti i formati di esportazione disponibili (AKN, XML, PDF, EPUB, RTF, URI/ELI, JSON).
Il campo `label` è il valore da usare nel parametro `formato` delle API asincrone.

In [3]:
status, estensioni = get("/bff-opendata/v1/api/v1/tipologiche/estensioni")
assert status == 200, f"Atteso 200, ricevuto {status}"
pp(estensioni)

# Estrai solo le label per uso successivo
formati_disponibili = [e["label"] for e in estensioni]
print("\nFormati disponibili:", formati_disponibili)

GET https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/tipologiche/estensioni  →  200
[
  {
    "label": "AKN",
    "value": "Esporta AKN"
  },
  {
    "label": "XML",
    "value": "Esporta XML"
  },
  {
    "label": "PDF",
    "value": "Esporta PDF"
  },
  {
    "label": "EPUB",
    "value": "Esporta EPUB"
  },
  {
    "label": "RTF",
    "value": "Esporta RTF"
  },
  {
    "label": "URI",
    "value": "Esporta ELI"
  },
  {
    "label": "JSON",
    "value": "Esporta JSON"
  },
  {
    "label": "HTML",
    "value": "Esporta HTML"
  }
]

Formati disponibili: ['AKN', 'XML', 'PDF', 'EPUB', 'RTF', 'URI', 'JSON', 'HTML']


### 1.2 Collezioni predefinite

`GET /bff-opendata/v1/api/v1/collections/collection-predefinite`

Elenca le collezioni preconfezionate disponibili (nome + numero di atti).
Il `nomeCollezione` è il valore da passare all'API di download (§ 6).

In [4]:
status, collezioni = get("/bff-opendata/v1/api/v1/collections/collection-predefinite")
assert status == 200, f"Atteso 200, ricevuto {status}"
pp(collezioni)

nomi_collezioni = [c["nomeCollezione"] for c in collezioni]
print("\nCollezioni disponibili:", nomi_collezioni)

GET https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/collections/collection-predefinite  →  200
[
  {
    "nomeCollezione": "Atti di attuazione Regolamenti UE",
    "formatoCollezione": "O",
    "descrizioneFormatoCollezione": "ORIGINALE",
    "dataCreazione": "2026-05-08",
    "numeroAtti": 39
  },
  {
    "nomeCollezione": "Atti di attuazione Regolamenti UE",
    "formatoCollezione": "M",
    "descrizioneFormatoCollezione": "MULTIVIGENTE",
    "dataCreazione": "2026-05-08",
    "numeroAtti": 39
  },
  {
    "nomeCollezione": "Atti di attuazione Regolamenti UE",
    "formatoCollezione": "V",
    "descrizioneFormatoCollezione": "VIGENTE",
    "dataCreazione": "2026-05-08",
    "numeroAtti": 39
  },
  {
    "nomeCollezione": "Atti di recepimento direttive UE",
    "formatoCollezione": "O",
    "descrizioneFormatoCollezione": "ORIGINALE",
    "dataCreazione": "2026-05-08",
    "numeroAtti": 1086
  },
  {
    "nomeCollezione": "Atti di recepimento direttive UE",
    "form

### 1.3 Ricerche predefinite

`GET /bff-opendata/v1/api/v1/ricerca/predefinita`

Restituisce le ricerche predefinite presenti nel portale (es. "Atti Repubblica", "Atti abrogati", ecc.).

In [5]:
status, ricerche_pred = get("/bff-opendata/v1/api/v1/ricerca/predefinita")
assert status == 200, f"Atteso 200, ricevuto {status}"
pp(ricerche_pred)

print("\nNomi ricerche predefinite:")
for r in ricerche_pred.get("ricerchePredefinite", []):
    print(" -", r["nome"])

GET https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/predefinita  →  200
{
  "ricerchePredefinite": [
    {
      "nome": "Atti Repubblica",
      "dettagli": [
        {
          "nomeCampo": "EmanazioneFrom",
          "valoreCampo": "1946-06-20"
        },
        {
          "nomeCampo": "EmanazioneTo",
          "valoreCampo": "2024-12-19"
        }
      ],
      "dataCreazione": "2024-12-17T20:23:02"
    },
    {
      "nome": "Atti Regno d'Italia aggiornati vigenti",
      "dettagli": [
        {
          "nomeCampo": "EmanazioneFrom",
          "valoreCampo": "1861-01-01"
        },
        {
          "nomeCampo": "EmanazioneTo",
          "valoreCampo": "1946-06-10"
        },
        {
          "nomeCampo": "classeProvvedimento",
          "valoreCampo": "2"
        }
      ],
      "dataCreazione": "2024-12-17T20:23:02"
    },
    {
      "nome": "Atti abrogati",
      "dettagli": [
        {
          "nomeCampo": "classeProvvedimento",
       

### 1.4 Classi di provvedimento

`GET /bff-opendata/v1/api/v1/tipologiche/classe-provvedimento`

Restituisce le classi di provvedimento:
- `1` → senza aggiornamenti
- `2` → aggiornato  
- `3` → abrogato

In [6]:
status, classi = get("/bff-opendata/v1/api/v1/tipologiche/classe-provvedimento")
assert status == 200, f"Atteso 200, ricevuto {status}"
pp(classi)

GET https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/tipologiche/classe-provvedimento  →  200
[
  {
    "label": "1",
    "value": "atto normativo – senza aggiornamenti"
  },
  {
    "label": "2",
    "value": "atto normativo – aggiornato"
  },
  {
    "label": "3",
    "value": "atto normativo – abrogato"
  }
]


### 1.5 Denominazione atto

`GET /bff-opendata/v1/api/v1/tipologiche/denominazione-atto`

Elenca tutti i tipi di provvedimento (LEGGE, DECRETO, DPR, ecc.).
Il campo `label` è il codice da usare come `codice_tipo_provvedimento` nei filtri.

In [7]:
status, denominazioni = get("/bff-opendata/v1/api/v1/tipologiche/denominazione-atto")
assert status == 200, f"Atteso 200, ricevuto {status}"
pp(denominazioni)

# Mappa codice → descrizione per uso successivo
codici_tipo = {d["label"]: d["value"] for d in denominazioni}
print(f"\nTipi di provvedimento disponibili: {len(codici_tipo)}")

GET https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/tipologiche/denominazione-atto  →  200
[
  {
    "label": "COS",
    "value": "COSTITUZIONE"
  },
  {
    "label": "DCT",
    "value": "DECRETO"
  },
  {
    "label": "PCG",
    "value": "DECRETO DEL CAPO DEL GOVERNO"
  },
  {
    "label": "3NA",
    "value": "DECRETO DEL CAPO DEL GOVERNO, PRIMO MINISTRO SEGRETARIO DI STATO"
  },
  {
    "label": "PCS",
    "value": "DECRETO DEL CAPO PROVVISORIO DELLO STATO"
  },
  {
    "label": "DDD",
    "value": "DECRETO DEL DUCE"
  },
  {
    "label": "FAC",
    "value": "DECRETO DEL DUCE DEL FASCISMO, CAPO DEL GOVERNO"
  },
  {
    "label": "PCM_DPC",
    "value": "DECRETO DEL PRESIDENTE DEL CONSIGLIO DEI MINISTRI"
  },
  {
    "label": "PPR",
    "value": "DECRETO DEL PRESIDENTE DELLA REPUBBLICA"
  },
  {
    "label": "PDL",
    "value": "DECRETO-LEGGE"
  },
  {
    "label": "DLL",
    "value": "DECRETO-LEGGE LUOGOTENENZIALE"
  },
  {
    "label": "PLL",
    "value": "DECRETO 

---
## 2. Ricerca sincrona

Le API di ricerca sincrona restituiscono immediatamente la lista di atti (metadati) con paginazione.

### 2.1 Ricerca Semplice — keyword nel testo

`POST /bff-opendata/v1/api/v1/ricerca/semplice`

Ricerca per parole chiave nel titolo e/o testo, con ordinamento e paginazione.

In [8]:
body_semplice = {
    "testoRicerca": "privacy",
    "orderType": "recente",
    "paginazione": {
        "paginaCorrente": 1,
        "numeroElementiPerPagina": 5
    }
}

status, res = post("/bff-opendata/v1/api/v1/ricerca/semplice", body_semplice)
assert status == 200, f"Atteso 200, ricevuto {status}"

print(f"Atti trovati: {res['numeroAttiTrovati']}  |  Pagine: {res['numeroPagine']}")
print("\nPrimi atti:")
for atto in res.get("listaAtti", []):
    print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/semplice  →  200
Atti trovati: 76  |  Pagine: 16

Primi atti:
  [2025] DECRETO LEGISLATIVO 30 dicembre 2025, n. 216
  [2025] DECRETO DEL PRESIDENTE DELLA REPUBBLICA 4 aprile 2025, n. 62
  [2024] DECRETO LEGISLATIVO 4 settembre 2024, n. 138
  [2024] DECRETO DEL PRESIDENTE DEL CONSIGLIO DEI MINISTRI 15 marzo 2024, n. 57
  [2023] DECRETO 4 aprile 2023, n. 59


In [9]:
# Esplora la facetMap (aggregazioni per anno, tipo, emittente)
print("FacetMap — distribuzione per anno provvedimento:")
for item in res.get("facetMap", {}).get("anno_provvedimento", []):
    print(f"  {item['descrizione']}: {item['valore']} atti")

print("\nFacetMap — distribuzione per tipo provvedimento:")
for item in res.get("facetMap", {}).get("codice_tipo_provvedimento", []):
    print(f"  {item['descrizione']}: {item['valore']} atti")

FacetMap — distribuzione per anno provvedimento:
  2003: 9 atti
  2004: 7 atti
  2022: 6 atti
  2006: 5 atti
  2011: 5 atti
  2015: 5 atti
  2016: 5 atti
  2020: 4 atti
  2001: 3 atti
  2007: 3 atti
  2010: 3 atti
  2012: 3 atti
  2018: 3 atti
  2021: 3 atti
  2005: 2 atti
  2024: 2 atti
  2025: 2 atti
  2000: 1 atti
  2002: 1 atti
  2013: 1 atti
  2017: 1 atti
  2019: 1 atti
  2023: 1 atti

FacetMap — distribuzione per tipo provvedimento:
  LEGGE: 37 atti
  DECRETO: 19 atti
  DECRETO DEL PRESIDENTE DELLA REPUBBLICA: 7 atti
  DECRETO LEGISLATIVO: 6 atti
  DECRETO-LEGGE: 4 atti
  None: 3 atti


### 2.1b Ricerca Semplice — paginazione alla pagina 2

In [10]:
body_pag2 = {
    "testoRicerca": "privacy",
    "orderType": "vecchio",  # dal meno recente
    "paginazione": {
        "paginaCorrente": 2,
        "numeroElementiPerPagina": 5
    }
}

status, res_p2 = post("/bff-opendata/v1/api/v1/ricerca/semplice", body_pag2)
print(f"Pagina corrente: {res_p2.get('paginaCorrente')}  |  Atti restituiti: {len(res_p2.get('listaAtti', []))}")
for atto in res_p2.get("listaAtti", []):
    print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/semplice  →  200
Pagina corrente: 2  |  Atti restituiti: 5
  [2003] LEGGE 20 marzo 2003, n. 74
  [2003] LEGGE 18 giugno 2003, n. 160
  [2003] LEGGE 18 giugno 2003, n. 164
  [2003] DECRETO LEGISLATIVO 30 giugno 2003, n. 196
  [2003] LEGGE 19 agosto 2003, n. 246


### 2.2 Ricerca Avanzata — filtri multipli

`POST /bff-opendata/v1/api/v1/ricerca/avanzata`

Permette di filtrare per: tipo atto, titolo, testo, date emanazione/pubblicazione, vigenza, classe provvedimento, anno/mese/giorno/numero provvedimento.

In [11]:
body_avanzata = {
    "denominazioneAtto": "LEGGE",
    "titoloRicerca": "ambiente",
    "dataInizioEmanazione": "2020-01-01",
    "dataFineEmanazione": "2023-12-31",
    "classeProvvedimento": "2",  # atti aggiornati
    "orderType": "recente",
    "paginazione": {
        "paginaCorrente": 1,
        "numeroElementiPerPagina": 5
    }
}

status, res_av = post("/bff-opendata/v1/api/v1/ricerca/avanzata", body_avanzata)
assert status == 200, f"Atteso 200, ricevuto {status}"

print(f"Atti trovati: {res_av['numeroAttiTrovati']}")
for atto in res_av.get("listaAtti", []):
    print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/avanzata  →  200
Atti trovati: 0


In [12]:
# Ricerca avanzata per numero e anno specifico del provvedimento
body_per_numero = {
    "denominazioneAtto": "DECRETO LEGISLATIVO",
    "annoProvvedimento": "2023",
    "numeroProvvedimento": "36",
    "orderType": "recente",
    "paginazione": {
        "paginaCorrente": 1,
        "numeroElementiPerPagina": 5
    }
}

status, res_num = post("/bff-opendata/v1/api/v1/ricerca/avanzata", body_per_numero)
print(f"Atti trovati: {res_num.get('numeroAttiTrovati')}")
for atto in res_num.get("listaAtti", []):
    print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")
    print(f"    Titolo: {atto['titoloAtto'][:100]}...")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/avanzata  →  200
Atti trovati: 1
  [2023] DECRETO LEGISLATIVO 31 marzo 2023, n. 36
    Titolo: [Codice dei contratti pubblici in  attuazione  dell'articolo  1  della legge 21 giugno 2022, n. 78, ...


### 2.3 Ricerca Semplice con FacetMap

Stesso endpoint della ricerca semplice ma con aggiunta del campo `filtriMap` per filtrare per tipo e/o anno.

In [13]:
body_facet_semplice = {
    "testoRicerca": "lavoro",
    "orderType": "recente",
    "paginazione": {
        "paginaCorrente": 1,
        "numeroElementiPerPagina": 5
    },
    "filtriMap": {
        "codice_tipo_provvedimento": "PLE",  # LEGGE
        "anno_provvedimento": 2023
    }
}

status, res_facet = post("/bff-opendata/v1/api/v1/ricerca/semplice", body_facet_semplice)
assert status == 200, f"Atteso 200, ricevuto {status}"

print(f"Atti trovati con filtri (LEGGE 2023): {res_facet['numeroAttiTrovati']}")
for atto in res_facet.get("listaAtti", []):
    print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/semplice  →  200
Atti trovati con filtri (LEGGE 2023): 53
  [2023] LEGGE 30 dicembre 2023, n. 223
  [2023] LEGGE 30 dicembre 2023, n. 214
  [2023] LEGGE 30 dicembre 2023, n. 213
  [2023] LEGGE 27 dicembre 2023, n. 206
  [2023] LEGGE 4 dicembre 2023, n. 202


### 2.4 Ricerca Avanzata con FacetMap

In [14]:
body_facet_avanzata = {
    "denominazioneAtto": "DECRETO",
    "titoloRicerca": "sicurezza",
    "dataInizioEmanazione": "2022-01-01",
    "dataFineEmanazione": "2022-12-31",
    "classeProvvedimento": "2",
    "orderType": "recente",
    "paginazione": {
        "paginaCorrente": 1,
        "numeroElementiPerPagina": 5
    },
    "filtriMap": {
        "codice_tipo_provvedimento": "DCT",
        "anno_provvedimento": 2022
    }
}

status, res_fav = post("/bff-opendata/v1/api/v1/ricerca/avanzata", body_facet_avanzata)
print(f"Atti trovati: {res_fav.get('numeroAttiTrovati')}")
for atto in res_fav.get("listaAtti", []):
    print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/avanzata  →  200
Atti trovati: 0


---
## 3. Ricerca asincrona — Export ZIP

Il flusso asincrono per esportare collezioni in ZIP si compone di tre passi:
1. **NuovaRicerca** → ottieni il token
2. **ConfermaRicerca** → conferma ed avvia l'elaborazione
3. **CheckStatus** (polling) → quando `status_code = 303` scarica lo ZIP dall'URL in `x-ipzs-location`

### 3.1 Nuova Ricerca asincrona (export)

`POST /bff-opendata/v1/api/v1/ricerca-asincrona/nuova-ricerca`

Restituisce un token (UUID stringa) da usare nei passi successivi.

In [15]:
body_nuova_ricerca = {
    "formato": "JSON",       # formato esportazione
    "tipoRicerca": "S",      # S = Semplice, A = Avanzata
    "modalita": "C",         # C = Classica, R = Responsive (opzionale)
    "parametriRicerca": {
        "testoRicerca": "costituzione",
        "filtriMap": {
            "codice_tipo_provvedimento": "COS"  # COSTITUZIONE
        }
    }
}

url_nuova = BASE_URL + "/bff-opendata/v1/api/v1/ricerca-asincrona/nuova-ricerca"
r_nuova = SESSION.post(url_nuova, headers=HEADERS_JSON, json=body_nuova_ricerca)
print(f"POST nuova-ricerca  →  {r_nuova.status_code}")

if r_nuova.status_code == 202:
    token = r_nuova.text.strip().strip('"')
    print(f"Token ricevuto: {token}")
elif r_nuova.status_code == 200:
    print("Risposta 200 — body vuoto (ricerca avanzata senza token nel body)")
    token = None
else:
    print(f"Errore: {r_nuova.text}")
    token = None

POST nuova-ricerca  →  202
Token ricevuto: 9f42dc49-52b2-4024-97aa-4263680c8622


### 3.2 Conferma Ricerca

`PUT /bff-opendata/v1/api/v1/ricerca-asincrona/conferma-ricerca`

Conferma la richiesta e avvia l'elaborazione asincrona.

In [16]:
if token:
    status, res_conf = put(
        "/bff-opendata/v1/api/v1/ricerca-asincrona/conferma-ricerca",
        {"token": token}
    )
    print(f"Stato conferma: {res_conf}")
    # stati attesi: 1 = in attesa, 2 = in elaborazione, 3 = completato con successo
    #              5 = carico eccessivo,  6 = possibile prolungamento tempi
else:
    print("Token non disponibile — salta conferma.")

PUT https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca-asincrona/conferma-ricerca  →  200
Stato conferma: {'stato': 1, 'descrizioneStato': 'Ricerca confermata in attesa di elaborazione', 'descrizioneErrore': None, 'totAtti': None, 'attiElaborati': None, 'percentuale': 0.0}


### 3.3 Check Status (polling) + Download ZIP

`GET /bff-opendata/v1/api/v1/ricerca-asincrona/check-status/<token>`

- `200` → continuare il polling
- `303` → URL di download nell'header `x-ipzs-location`

In [17]:
POLL_INTERVAL_S = 5
MAX_ATTEMPTS   = 12

download_url = None

if token:
    check_path = f"/bff-opendata/v1/api/v1/ricerca-asincrona/check-status/{token}"
    for attempt in range(1, MAX_ATTEMPTS + 1):
        url = BASE_URL + check_path
        r = SESSION.get(url, headers=HEADERS, allow_redirects=False)
        print(f"  Tentativo {attempt}: HTTP {r.status_code}")

        if r.status_code == 303:
            download_url = r.headers.get("x-ipzs-location")
            print(f"  Pronto! URL download: {download_url}")
            break
        elif r.status_code == 200:
            try:
                body = r.json()
                print(f"    Stato: {body.get('stato')} — {body.get('descrizioneStato')}")
            except Exception:
                pass
            time.sleep(POLL_INTERVAL_S)
        else:
            print(f"  Errore inatteso: {r.text}")
            break
    else:
        print("  Timeout: elaborazione non completata entro il numero massimo di tentativi.")
else:
    print("Token non disponibile — salta check status.")

  Tentativo 1: HTTP 200
    Stato: 1 — Ricerca confermata in attesa di elaborazione
  Tentativo 2: HTTP 200
    Stato: 1 — Ricerca confermata in attesa di elaborazione
  Tentativo 3: HTTP 303
  Pronto! URL download: https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/collections/download/collection-asincrona/9f42dc49-52b2-4024-97aa-4263680c8622


In [20]:
import os



# Download effettivo del file ZIP (se l'URL è disponibile)
if download_url:
    r_zip = SESSION.get(download_url, headers=HEADERS)
    print(f"Download ZIP  →  HTTP {r_zip.status_code}")
    if r_zip.status_code == 200:
        zip_path = "/home/adellorto/projects/normattiva-mcp/tests/output/normattiva_export.zip"
        os.makedirs(os.path.dirname(zip_path), exist_ok=True)
        with open(zip_path, "wb") as f:
            f.write(r_zip.content)
        print(f"File salvato in: {zip_path}  ({len(r_zip.content):,} bytes)")
    else:
        print(f"Errore download: {r_zip.text}")
else:
    print("URL di download non disponibile.")

Download ZIP  →  HTTP 200
File salvato in: /home/adellorto/projects/normattiva-mcp/tests/output/normattiva_export.zip  (25,816 bytes)


---
## 4. Dettaglio atto — navigazione per articolo

`POST /bff-opendata/v1/api/v1/atto/dettaglio-atto`

Recupera il contenuto HTML di un articolo specifico dato il codice redazionale, l'id articolo e la data di vigenza.

> **Nota:** `dataGU` e `dataVigenza` usano il formato `YYYY-MM-DD` (nonostante la doc indichi `YYYY-DD-MM`).

In [21]:
# Esempio: LEGGE 400/1988 (Legge sull'ordinamento della Presidenza del Consiglio)
# art. 13-bis (id articolo 13, sotto articolo 2)
body_dettaglio = {
    "dataGU": "1988-09-12",       # data pubblicazione in GU
    "codiceRedazionale": "088G0458",
    "idArticolo": 13,
    "sottoArticolo": 2,
    "sottoArticolo1": 0,
    "dataVigenza": "2025-01-01",   # data di vigenza richiesta
    "idGruppo": 6,
    "progressivo": 0,
    "versione": 0                  # 0 = versione originale
}

status, res_det = post("/bff-opendata/v1/api/v1/atto/dettaglio-atto", body_dettaglio)
print(f"HTTP status: {status}")

if status == 200 and res_det.get("success"):
    atto = res_det["data"]["atto"]
    print(f"Titolo: {atto['titolo']}")
    print(f"Tipo: {atto['tipoProvvedimentoDescrizione']} ({atto['tipoProvvedimentoCodice']})")
    print(f"Vigenza articolo: {atto['articoloDataInizioVigenza']} → {atto['articoloDataFineVigenza']}")
    html = atto.get('articoloHtml', '')
    print(f"\nArticoloHtml (primi 500 char):\n{html[:500]}")
else:
    pp(res_det)

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/atto/dettaglio-atto  →  200
HTTP status: 200
Titolo: LEGGE 23 agosto 1988, n. 400
Tipo: LEGGE (PLE)
Vigenza articolo: 20090704 → 99999999

ArticoloHtml (primi 500 char):
<div class="bodyTesto">
        <h2 class="preamble-title-akn"></h2>
        <h2 class="preamble-end-akn"></h2>
        <h2 class="article-num-akn" id="art_13-bis">Art. 13-bis</h2>
        <div class="article-pre-comma-text-akn">
            <div class="ins-akn" eId="ins_1">(( (Chiarezza dei testi normativi). ))</div>
             
            <br>
             
            <br>
             ((
        </div>
        <div class="art-commi-div-akn">
            <div class="art-comma-div-akn">
   


In [22]:
# Articolo 1 della stessa legge (versione originale)
body_det_art1 = {
    "dataGU": "1988-09-12",
    "codiceRedazionale": "088G0458",
    "idArticolo": 1,
    "sottoArticolo": 0,
    "sottoArticolo1": 0,
    "dataVigenza": "2025-01-01",
    "idGruppo": 0,
    "progressivo": 0,
    "versione": 0
}

status, res_det1 = post("/bff-opendata/v1/api/v1/atto/dettaglio-atto", body_det_art1)
print(f"HTTP status: {status}")
if status == 200 and res_det1.get("success"):
    atto1 = res_det1["data"]["atto"]
    print(f"Titolo: {atto1['titolo']}")
    print(f"HTML (primi 300 char): {atto1.get('articoloHtml', '')[:300]}")
else:
    pp(res_det1)

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/atto/dettaglio-atto  →  404
HTTP status: 404
{
  "message": "dataPubblicazioneGazzetta:Mon Sep 12 00:00:00 CEST 1988 codiceRedazionale:088G0458 idArticolo:1 idSottoArticolo:0 idSottoArticolo1:null flagTipoArticolo:0",
  "code": null
}


---
## 5. Download collezione preconfezionata

`GET /bff-opendata/v1/api/v1/collections/download/collection-preconfezionata`

Scarica un file ZIP con gli atti di una collezione predefinita nel formato scelto.

**Parametri:**
- `nome`: nome della collezione (da § 1.2)
- `formato`: formato (AKN, EPUB, HTML, JSON, PDF, RTF, XML, URI)
- `formatoRichiesta`: `O` (originario), `V` (vigente), `M` (multivigente)

In [ ]:
# Prima verifichiamo i nomi esatti delle collezioni disponibili (da § 1.2)
if nomi_collezioni:
    nome_collezione = nomi_collezioni[0]  # usa la prima disponibile
    print(f"Collezione scelta: {nome_collezione}")
else:
    nome_collezione = "Codici"  # fallback
    print(f"Nessuna collezione recuperata, uso il fallback: {nome_collezione}")

In [ ]:
# Download in formato JSON, versione vigente
params_coll = {
    "nome": nome_collezione,
    "formato": "JSON",
    "formatoRichiesta": "V"  # Vigente
}

url_coll = BASE_URL + "/bff-opendata/v1/api/v1/collections/download/collection-preconfezionata"
r_coll = SESSION.get(url_coll, headers=HEADERS, params=params_coll)
print(f"GET download collection  →  HTTP {r_coll.status_code}")
print(f"Content-Type: {r_coll.headers.get('Content-Type')}")
print(f"Content-Length: {r_coll.headers.get('Content-Length', 'n/d')} bytes")

if r_coll.status_code == 200:
    out_path = f"/tmp/normattiva_collection.zip"
    with open(out_path, "wb") as f:
        f.write(r_coll.content)
    print(f"File salvato: {out_path}  ({len(r_coll.content):,} bytes)")
elif r_coll.status_code in (400, 404, 500):
    try:
        pp(r_coll.json())
    except Exception:
        print(r_coll.text)

In [ ]:
# Prova formato non valido — atteso errore 400 con code 1003
params_bad_fmt = {
    "nome": nome_collezione,
    "formato": "DOCX",     # formato non supportato
    "formatoRichiesta": "V"
}
r_bad = SESSION.get(url_coll, headers=HEADERS, params=params_bad_fmt)
print(f"Formato non valido  →  HTTP {r_bad.status_code}")
try:
    pp(r_bad.json())
except Exception:
    print(r_bad.text)

In [ ]:
# Prova formatoRichiesta non valido — atteso errore 400 con code 1006
params_bad_vign = {
    "nome": nome_collezione,
    "formato": "JSON",
    "formatoRichiesta": "X"  # valore non ammesso (solo O, M, V)
}
r_bad_v = SESSION.get(url_coll, headers=HEADERS, params=params_bad_vign)
print(f"FormatoRichiesta non valido  →  HTTP {r_bad_v.status_code}")
try:
    pp(r_bad_v.json())
except Exception:
    print(r_bad_v.text)

---
## 6. Atti aggiornati tra due date

`POST /bff-opendata/v1/api/v1/ricerca/aggiornati`

Restituisce la lista di atti che sono stati modificati/aggiornati in un intervallo temporale.

**Limiti:**
- Massimo 12 mesi tra le due date (errore 400 code `1501` se superato)
- Massimo 7000 atti (errore 400 code `1502`)
- La data finale non può precedere quella iniziale (errore 400 code `1503`)

In [ ]:
# Atti aggiornati in una settimana recente
data_fine   = datetime(2024, 4, 29)
data_inizio = data_fine - timedelta(days=7)

body_aggiornati = {
    "dataInizioAggiornamento": data_inizio.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
    "dataFineAggiornamento":   data_fine.strftime("%Y-%m-%dT%H:%M:%S.000Z")
}

status, res_agg = post("/bff-opendata/v1/api/v1/ricerca/aggiornati", body_aggiornati)
print(f"HTTP status: {status}")

if status == 200:
    print(f"Atti aggiornati trovati: {res_agg.get('numeroAttiTrovati')}")
    for atto in res_agg.get("listaAtti", [])[:5]:
        print(f"  [{atto['annoProvvedimento']}] {atto['descrizioneAtto']}")
        print(f"    Ultima modifica: {atto.get('dataUltimaModifica')}  —  Modificato da: {atto.get('ultimiAttiModificanti')}")
else:
    pp(res_agg)

In [ ]:
# Errore 400 code 1501: intervallo > 12 mesi
body_over_12m = {
    "dataInizioAggiornamento": "2022-01-01T00:00:00.000Z",
    "dataFineAggiornamento":   "2024-01-01T00:00:00.000Z"  # 24 mesi
}
status_err, res_err = post("/bff-opendata/v1/api/v1/ricerca/aggiornati", body_over_12m)
print(f"Intervallo > 12 mesi  →  HTTP {status_err}")
pp(res_err)

In [ ]:
# Errore 400 code 1503: data finale precede quella iniziale
body_inversione = {
    "dataInizioAggiornamento": "2024-04-29T00:00:00.000Z",
    "dataFineAggiornamento":   "2024-04-01T00:00:00.000Z"  # fine < inizio
}
status_inv, res_inv = post("/bff-opendata/v1/api/v1/ricerca/aggiornati", body_inversione)
print(f"Date invertite  →  HTTP {status_inv}")
pp(res_inv)

---
## 7. Gestione degli errori

Verifica le risposte di errore standard documentate (§ 1.3.10).

In [ ]:
# 404 — endpoint inesistente
status_404, res_404 = get("/bff-opendata/v1/api/v1/endpoint-inesistente")
print(f"Endpoint inesistente  →  HTTP {status_404}")
pp(res_404)

In [ ]:
# 400 — body malformato (ricerca semplice senza campo obbligatorio)
status_400, res_400 = post("/bff-opendata/v1/api/v1/ricerca/semplice", {})
print(f"Body malformato  →  HTTP {status_400}")
pp(res_400)

In [23]:
# 404 custom — dettaglio atto con codice redazionale inesistente
body_not_found = {
    "dataGU": "2000-01-01",
    "codiceRedazionale": "XXXXXXXXXX",
    "idArticolo": 1,
    "sottoArticolo": 0,
    "sottoArticolo1": 0,
    "dataVigenza": "2025-01-01",
    "idGruppo": 0,
    "progressivo": 0,
    "versione": 0
}
status_nf, res_nf = post("/bff-opendata/v1/api/v1/atto/dettaglio-atto", body_not_found)
print(f"Atto inesistente  →  HTTP {status_nf}")
pp(res_nf)

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/atto/dettaglio-atto  →  404
Atto inesistente  →  HTTP 404
{
  "message": "Atto non trovato",
  "code": null
}


---
## 8. Scenari di utilizzo reale

Esempi pratici di query utili per un progetto MCP o RAG su normativa italiana.

In [24]:
# Scenario A: Trovare tutte le Leggi sul GDPR/privacy degli ultimi 5 anni
body_gdpr = {
    "denominazioneAtto": "DECRETO LEGISLATIVO",
    "titoloRicerca": "protezione dei dati",
    "dataInizioEmanazione": "2018-01-01",
    "dataFineEmanazione": "2024-12-31",
    "classeProvvedimento": "2",
    "orderType": "recente",
    "paginazione": {"paginaCorrente": 1, "numeroElementiPerPagina": 10}
}
status, res_gdpr = post("/bff-opendata/v1/api/v1/ricerca/avanzata", body_gdpr)
print(f"Decreti Legislativi su 'protezione dei dati' 2018-2024: {res_gdpr.get('numeroAttiTrovati')} trovati")
for a in res_gdpr.get("listaAtti", []):
    print(f"  {a['descrizioneAtto']}")

POST https://api.normattiva.it/t/normattiva.api/bff-opendata/v1/api/v1/ricerca/avanzata  →  200
Decreti Legislativi su 'protezione dei dati' 2018-2024: 37 trovati
  DECRETO LEGISLATIVO 13 dicembre 2024, n. 192
  DECRETO LEGISLATIVO 25 novembre 2024, n. 190
  DECRETO LEGISLATIVO 5 novembre 2024, n. 174
  DECRETO LEGISLATIVO 10 settembre 2024, n. 147
  DECRETO LEGISLATIVO 28 marzo 2024, n. 45
  DECRETO LEGISLATIVO 5 febbraio 2024, n. 20
  DECRETO LEGISLATIVO 13 dicembre 2023, n. 222
  DECRETO LEGISLATIVO 30 novembre 2023, n. 175
  DECRETO LEGISLATIVO 17 marzo 2023, n. 42
  DECRETO LEGISLATIVO 31 marzo 2023, n. 36


In [ ]:
# Scenario B: Atti del Ministero della Salute negli ultimi 2 anni
body_salute = {
    "testoRicerca": "salute",
    "orderType": "recente",
    "paginazione": {"paginaCorrente": 1, "numeroElementiPerPagina": 10},
    "filtriMap": {"anno_provvedimento": 2024}
}
status, res_sal = post("/bff-opendata/v1/api/v1/ricerca/semplice", body_salute)
print(f"Atti 2024 con 'salute': {res_sal.get('numeroAttiTrovati')}")
emittenti = res_sal.get("facetMap", {}).get("descrizione_emettitore", [])
print("Top emittenti:")
for e in sorted(emittenti, key=lambda x: x["valore"], reverse=True)[:5]:
    print(f"  {e['descrizione']}: {e['valore']} atti")

In [ ]:
# Scenario C: Monitoraggio modifiche normative settimanali (sistema di alerting)
oggi = datetime.now()
settimana_fa = oggi - timedelta(days=7)

body_monitor = {
    "dataInizioAggiornamento": settimana_fa.strftime("%Y-%m-%dT00:00:00.000Z"),
    "dataFineAggiornamento":   oggi.strftime("%Y-%m-%dT00:00:00.000Z")
}
status, res_mon = post("/bff-opendata/v1/api/v1/ricerca/aggiornati", body_monitor)
print(f"Atti modificati nell'ultima settimana: {res_mon.get('numeroAttiTrovati', 0)}")
if status == 200:
    for a in res_mon.get("listaAtti", [])[:5]:
        print(f"  {a['descrizioneAtto']} — modificato il {a.get('dataUltimaModifica')}")

In [ ]:
# Scenario D: Ricerca Costituzione e recupero articolo specifico
# Passo 1 — trova la Costituzione
body_cost = {
    "denominazioneAtto": "COSTITUZIONE",
    "orderType": "vecchio",
    "paginazione": {"paginaCorrente": 1, "numeroElementiPerPagina": 5}
}
status, res_cost = post("/bff-opendata/v1/api/v1/ricerca/avanzata", body_cost)
print(f"Risultati per 'COSTITUZIONE': {res_cost.get('numeroAttiTrovati')}")
for a in res_cost.get("listaAtti", []):
    print(f"  codiceRedazionale={a.get('codiceRedazionale')}  dataGU={a.get('dataGU')}  →  {a['descrizioneAtto']}")

In [ ]:
# Passo 2 — recupera articolo 1 della Costituzione
# (codiceRedazionale e dataGU presi dal risultato precedente)
atti_cost = res_cost.get("listaAtti", [])

if atti_cost:
    primo = atti_cost[0]
    body_cost_art1 = {
        "dataGU": primo["dataGU"],
        "codiceRedazionale": primo["codiceRedazionale"],
        "idArticolo": 1,
        "sottoArticolo": 0,
        "sottoArticolo1": 0,
        "dataVigenza": datetime.now().strftime("%Y-%m-%d"),
        "idGruppo": 0,
        "progressivo": 0,
        "versione": 0
    }
    status, res_c1 = post("/bff-opendata/v1/api/v1/atto/dettaglio-atto", body_cost_art1)
    print(f"HTTP {status}")
    if status == 200 and res_c1.get("success"):
        art = res_c1["data"]["atto"]
        print(f"Titolo: {art['titolo']}")
        print(f"HTML articolo:\n{art.get('articoloHtml', '')[:600]}")
    else:
        pp(res_c1)
else:
    print("Nessun atto trovato — salta recupero articolo.")

---
*Fine notebook — Normattiva OpenData API Tests*